# Tablas para los gráficos

## Objetivo

En este notebook armamos todas las tablas que usan los gráficos y las
guardamos en `data/tablas/`.



## 1. Importacion de librerias

In [1]:
import pandas as pd
from config import con, q, guardar, ruta_tablas, FEATURES, MIN_ESCUCHAS, MIN_USUARIOS_PAIS

print("Features:", FEATURES)
print("Mínimo de escuchas por usuario:", MIN_ESCUCHAS)
print("Mínimo de usuarios por país:", MIN_USUARIOS_PAIS)

Features: ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'speechiness', 'liveness', 'tempo_norm', 'loudness_norm']
Mínimo de escuchas por usuario: 100
Mínimo de usuarios por país: 50


## 2. Tabla por usuario

Tiene **una fila por usuario**:

| Columna | De dónde sale | Operación |
|---|---|---|
| `user_id`, `country`, `age`, `gender` | `users_clean` | |
| `franja_edad` | se calcula a partir de `age` | Derivación |
| `n_escuchas` | cantidad de escuchas del usuario | Agregación (conteo) |
| `danceability`, `energy`, `valence`, `acousticness`, `instrumentalness`, `speechiness`, `liveness` | promedio de cada feature en las canciones que escuchó el usuario | Agregación (promedio) |
| `tempo_norm`, `loudness_norm` | promedio de `tempo` y de `loudness`, llevados a una escala de 0 a 1 | Derivación (normalización) y agregación |

In [2]:
#`tempo` está en BPM y `loudness` en dB (valores negativos), así que no se
#pueden comparar directo con las otras.
#Normalizamos con el metodo mínimo y máximo de tempo y loudness en las canciones limpias (los nulos no cuentan)
r = q("""
    SELECT MIN(tempo) AS tempo_min, MAX(tempo) AS tempo_max,
           MIN(loudness) AS loud_min, MAX(loudness) AS loud_max
    FROM acoustic_features_clean
""").iloc[0]
tempo_min, tempo_max, loud_min, loud_max = r["tempo_min"], r["tempo_max"], r["loud_min"], r["loud_max"]
print(f"tempo:    de {tempo_min:.1f} a {tempo_max:.1f} BPM")
print(f"loudness: de {loud_min:.1f} a {loud_max:.1f} dB")

tempo:    de 30.1 a 250.0 BPM
loudness: de -59.8 a 6.6 dB


-Calculamos primero los promedios de las características musicales por
usuario. Si calculáramos directamente el promedio por país, los usuarios
con más escuchas tendrían más peso que los demás.

-Unimos las escuchas con las características de audio a través de track_id
y agrupamos por usuario para calcular la cantidad de escuchas y los
promedios. En el caso de tempo y loudness, primero normalizamos los valores
y después calculamos el promedio.

Con un JOIN a users_clean agregamos los datos de cada usuario: país,
edad y género. Solo contamos las escuchas de canciones que tienen un registro en la
tabla de características de audio. Por lo tanto, el mínimo de escuchas
que pedimos por usuario se aplica sobre esas escuchas y no sobre el total.

-Dividimos a los usuarios en cuatro franjas de edad: 10-19, 20-24, 25-29
y 30+. Para definir los rangos tuvimos en cuenta la distribución de las
edades que vimos en la exploración, donde la mayor parte de los usuarios
se concentra en edades jóvenes. Los usuarios sin edad válida quedan
con la franja nula.

-Por último, filtramos los usuarios con pocas escuchas

In [3]:
guardar(f"""
    SELECT u.user_id, u.country, u.age, u.gender,
           CASE WHEN u.age < 20 THEN '10-19'
                WHEN u.age < 25 THEN '20-24'
                WHEN u.age < 30 THEN '25-29'
                WHEN u.age >= 30 THEN '30+'
           END AS franja_edad,
           p.n_escuchas,
           p.danceability, p.energy, p.valence, p.acousticness,
           p.instrumentalness, p.speechiness, p.liveness,
           p.tempo_norm, p.loudness_norm
    FROM users_clean u
    JOIN (
        SELECT e.user_id,
               COUNT(*)              AS n_escuchas,
               AVG(f.danceability)      AS danceability,
               AVG(f.energy)            AS energy,
               AVG(f.valence)           AS valence,
               AVG(f.acousticness)      AS acousticness,
               AVG(f.instrumentalness)  AS instrumentalness,
               AVG(f.speechiness)       AS speechiness,
               AVG(f.liveness)          AS liveness,
               AVG((f.tempo - {tempo_min}) / ({tempo_max} - {tempo_min}))      AS tempo_norm,
               AVG((f.loudness - {loud_min}) / ({loud_max} - {loud_min}))      AS loudness_norm
        FROM events_clean e
        JOIN acoustic_features_clean f ON e.track_id = f.track_id
        GROUP BY e.user_id
    ) p ON u.user_id = p.user_id
    WHERE p.n_escuchas >= {MIN_ESCUCHAS}
""", "tabla_usuario")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

tabla_usuario: 53,289 filas guardadas en C:/Users/mgrau/musica-por-pais/data/tablas/tabla_usuario.parquet


In [4]:
# Ruta de la tabla que armamos
tabla = ruta_tablas + "tabla_usuario.parquet"

# Contamos los usuarios antes y después de aplicar el filtro de escuchas
total = q("SELECT COUNT(*) AS n FROM users_clean")["n"][0]
quedan = q(f"SELECT COUNT(*) AS n FROM '{tabla}'")["n"][0]

print(f"Usuarios en users_clean: {total:,}")
print(f"Usuarios en la tabla:    {quedan:,}")
print(f"Quedaron afuera:         {total - quedan:,} (menos de {MIN_ESCUCHAS} escuchas con datos de audio)")

# Contamos cuántos usuarios hay en cada franja de edad.
# Los que no informaron su edad o tenían una edad fuera del rango quedan como nulos.
display(q(f"SELECT franja_edad, COUNT(*) AS usuarios FROM '{tabla}' GROUP BY franja_edad ORDER BY franja_edad"))

# Contamos los usuarios por género y ordenamos de mayor a menor cantidad
display(q(f"SELECT gender, COUNT(*) AS usuarios FROM '{tabla}' GROUP BY gender ORDER BY usuarios DESC"))

# Seleccionamos la cantidad de escuchas y las características de audio
datos = q(f"SELECT n_escuchas, {', '.join(FEATURES)} FROM '{tabla}'")

# Revisamos los estadísticos para controlar el mínimo de escuchas
# y que los promedios de las características estén entre 0 y 1.
# Redondeamos los resultados a 3 decimales.
display(datos.describe().round(3))

# Contamos los nulos de cada columna.
# Esperamos encontrar nulos únicamente en edad y franja de edad.
display(q(f"SELECT * FROM '{tabla}'").isna().sum())

Usuarios en users_clean: 55,190
Usuarios en la tabla:    53,289
Quedaron afuera:         1,901 (menos de 100 escuchas con datos de audio)


,franja_edad,usuarios
0,10-19,8061
1,20-24,17001
2,25-29,8556
3,30+,7710
4,None,11961


,gender,usuarios
0,m,35346
1,f,13391
2,no informado,4552


,n_escuchas,danceability,energy,valence,acousticness,instrumentalness,speechiness,liveness,tempo_norm,loudness_norm
count,53289.000,53289.000,53289.000,53289.000,53289.000,53289.000,53289.000,53289.000,53289.000,53289.000
mean,6580.227,0.496,0.693,0.424,0.211,0.215,0.076,0.203,0.426,0.781
std,9081.736,0.079,0.117,0.097,0.132,0.153,0.030,0.027,0.024,0.031
min,100.000,0.104,0.103,0.044,0.000,0.000,0.030,0.083,0.242,0.540
25%,1534.000,0.445,0.615,0.362,0.108,0.094,0.058,0.185,0.412,0.763
50%,5310.000,0.500,0.694,0.430,0.192,0.181,0.069,0.200,0.426,0.784
75%,8865.000,0.548,0.779,0.489,0.289,0.306,0.086,0.217,0.440,0.803
max,345014.000,0.862,0.965,0.845,0.981,0.935,0.815,0.729,0.630,0.858


user_id                 0
country                 0
age                 11961
gender                  0
franja_edad         11961
n_escuchas              0
danceability            0
energy                  0
valence                 0
acousticness            0
instrumentalness        0
speechiness             0
liveness                0
tempo_norm              0
loudness_norm           0
dtype: int64

## 3. Tablas por país y por edad

### 3.1 Promedio por país

- Agregación: agrupamos los usuarios por país, calculamos el promedio
  de cada característica musical y contamos cuántos usuarios hay.
  Como cada usuario ya tiene sus propios promedios, todos tienen
  el mismo peso.

- Filtrado: nos quedamos con los países que tienen al menos
  MIN_USUARIOS_PAIS usuarios. Decidimos aplicar este filtro porque
  con pocos usuarios el promedio puede no representar bien los
  gustos musicales del país.

In [5]:
# Armamos un AVG por cada característica de la lista FEATURES
# para no tener que escribirlos uno por uno
promedios = ",\n           ".join(f"AVG({f}) AS {f}" for f in FEATURES)

# Agrupamos los usuarios por país, contamos cuántos hay
# y calculamos el promedio de cada característica musical.
# Nos quedamos con los países que cumplen con el mínimo de usuarios establecido.
guardar(f"""
    SELECT country,
           COUNT(*) AS n_usuarios,
           {promedios}
    FROM tabla_usuario
    GROUP BY country
    HAVING COUNT(*) >= {MIN_USUARIOS_PAIS}
""", "tabla_pais")

tabla_pais: 56 filas guardadas en C:/Users/mgrau/musica-por-pais/data/tablas/tabla_pais.parquet


In [6]:
# Cargamos la tabla que armamos por país
paises = q(f"SELECT * FROM '{ruta_tablas}tabla_pais.parquet'")

# Mostramos cuántos países quedaron después del filtro
# y sumamos la cantidad de usuarios de esos países
print("Países:", len(paises), "| Usuarios que representan:", f"{paises['n_usuarios'].sum():,}")

# Buscamos el valor mínimo y máximo de cada característica musical entre los países.
# Usamos .T para que cada característica quede en una fila.
resumen = paises[FEATURES].agg(["min", "max"]).T

# Calculamos la diferencia entre el máximo y el mínimo
# para ver en qué características hay más diferencias entre países
resumen["diferencia"] = resumen["max"] - resumen["min"]

resumen.round(3)

Países: 56 | Usuarios que representan: 52,098


,min,max,diferencia
danceability,0.443,0.537,0.094
energy,0.578,0.741,0.163
valence,0.308,0.462,0.155
acousticness,0.152,0.331,0.179
instrumentalness,0.120,0.319,0.199
speechiness,0.061,0.082,0.021
liveness,0.184,0.210,0.027
tempo_norm,0.412,0.434,0.022
loudness_norm,0.754,0.794,0.040



Las mayores diferencias entre países se dan en instrumentalness,
acousticness y energy. En cambio, speechiness y liveness tienen
poca diferencia entre sus valores mínimos y máximos.

### 3.2 Promedio por franja de edad

**Agregación** por franja de edad. Dejamos afuera a los usuarios sin edad válida,
porque no se pueden ubicar en ninguna franja.

In [7]:
# Agrupamos los usuarios por franja de edad, contamos cuántos hay
# en cada una y calculamos el promedio de cada característica musical.
# No tenemos en cuenta a los usuarios que no tienen una franja asignada.
guardar(f"""
    SELECT franja_edad,
           COUNT(*) AS n_usuarios,
           {promedios}
    FROM tabla_usuario
    WHERE franja_edad IS NOT NULL
    GROUP BY franja_edad
    ORDER BY franja_edad
""", "tabla_edad")

q(f"SELECT * FROM '{ruta_tablas}tabla_edad.parquet'").round(3)

tabla_edad: 4 filas guardadas en C:/Users/mgrau/musica-por-pais/data/tablas/tabla_edad.parquet


,franja_edad,n_usuarios,danceability,energy,valence,acousticness,instrumentalness,speechiness,liveness,tempo_norm,loudness_norm
0,10-19,8061,0.493,0.733,0.427,0.171,0.165,0.082,0.207,0.433,0.794
1,20-24,17001,0.489,0.709,0.416,0.194,0.210,0.078,0.204,0.429,0.786
2,25-29,8556,0.495,0.688,0.419,0.214,0.234,0.075,0.201,0.426,0.778
3,30+,7710,0.509,0.649,0.441,0.255,0.232,0.070,0.199,0.420,0.767


### 3.3 País con cultura y felicidad


- Agregación: como la tabla de happiness tiene datos por país y por año,
  calculamos el promedio de life_ladder de los años disponibles entre
  2005 y 2014. Así nos queda un solo valor por país.

- Combinación: unimos tabla_pais con los datos de Hofstede y happiness
  mediante el código de país. Usamos JOIN para quedarnos únicamente
  con los países que aparecen en las tres tablas.

In [8]:
# hacemos un join de los datos musicales de cada país con las dimensiones culturales
# de Hofstede y el indicador de felicidad.

guardar("""
    SELECT p.*,
           h.country_name,
           h.power_distance, h.individualism, h.masculinity,
           h.uncertainty_avoidance, h.long_term_orientation, h.indulgence,
           f.life_ladder
    FROM tabla_pais p

    -- Incorporamos las seis dimensiones culturales de Hofstede
    JOIN hofstede_clean h ON p.country = h.country

    -- Calculamos el promedio del indicador de felicidad por país
    JOIN (SELECT country, AVG(life_ladder) AS life_ladder
          FROM happiness_clean
          GROUP BY country) f ON p.country = f.country

""", "tabla_pais_cultura")

tabla_pais_cultura: 47 filas guardadas en C:/Users/mgrau/musica-por-pais/data/tablas/tabla_pais_cultura.parquet


In [9]:
cultura = q(f"SELECT * FROM '{ruta_tablas}tabla_pais_cultura.parquet'")
print("Países:", len(cultura))

# Países de Hofstede que quedaron afuera (por tener pocos usuarios)
display(q(f"""
    SELECT country, country_name FROM hofstede_clean
    WHERE country NOT IN (SELECT country FROM '{ruta_tablas}tabla_pais_cultura.parquet')
"""))

# Nulos (algunas dimensiones de Hofstede no están para todos los países)
cultura.isna().sum()

Países: 47


,country,country_name


country                  0
n_usuarios               0
danceability             0
energy                   0
valence                  0
acousticness             0
instrumentalness         0
speechiness              0
liveness                 0
tempo_norm               0
loudness_norm            0
country_name             0
power_distance           2
individualism            2
masculinity              2
uncertainty_avoidance    2
long_term_orientation    0
indulgence               1
life_ladder              0
dtype: int64

## 4. Tablas de artistas

### 4.1 Escuchas por usuario y artista

In [10]:
# Calculamos la cantidad de escuchas de cada usuario por artista.
guardar("""
    SELECT user_id, artist_id, COUNT(*) AS escuchas
    FROM events_clean

    -- Nos quedamos con los usuarios incluidos en tabla_usuario
    WHERE user_id IN (SELECT user_id FROM tabla_usuario)

    -- Agrupamos por usuario y artista para contar las escuchas
    GROUP BY user_id, artist_id
""", "usuario_artista")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

usuario_artista: 17,978,113 filas guardadas en C:/Users/mgrau/musica-por-pais/data/tablas/usuario_artista.parquet


### 4.2 Artistas por país

Para comparar los gustos musicales entre países, armamos una tabla con una fila por país y artista.

Usamos la cantidad de oyentes distintos en lugar de la cantidad de escuchas, para evitar que un usuario que escucha muchas veces al mismo artista tenga demasiado peso en el resultado.

Calculamos qué porcentaje de los usuarios de cada país escucha a cada artista. Así podemos comparar los gustos musicales entre países sin que influya la diferencia en la cantidad de usuarios.

Filtrado: nos quedamos con los N_ARTISTAS artistas con mayor cantidad de oyentes a nivel global. De esta forma, reducimos la cantidad de datos a analizar y dejamos afuera a los artistas con muy pocos oyentes.

In [11]:
N_ARTISTAS = 10000   # cantidad de artistas con más oyentes que usamos para comparar países

# Armamos una tabla con una fila por país y artista
# para comparar los gustos musicales entre países.

guardar(f"""
    -- Obtenemos los usuarios y su país
    WITH usuarios AS (
        SELECT user_id, country FROM tabla_usuario
    ),

    -- Contamos los usuarios por país y nos quedamos con los países
    -- que cumplen con el mínimo de usuarios definido
    usuarios_pais AS (
        SELECT country, COUNT(*) AS n FROM usuarios
        GROUP BY country HAVING COUNT(*) >= {MIN_USUARIOS_PAIS}
    ),

    -- Contamos cuántos usuarios de cada país escucharon a cada artista
    oyentes_pais AS (
        SELECT u.country, ua.artist_id, COUNT(*) AS oyentes
        FROM '{ruta_tablas}usuario_artista.parquet' ua
        JOIN usuarios u ON ua.user_id = u.user_id
        GROUP BY u.country, ua.artist_id
    ),

    -- Seleccionamos los N_ARTISTAS con mayor cantidad de oyentes en total
    mas_oyentes AS (
        SELECT artist_id FROM oyentes_pais
        GROUP BY artist_id ORDER BY SUM(oyentes) DESC LIMIT {N_ARTISTAS}
    )

    -- Dividimos la cantidad de oyentes del artista por el total de usuarios
    -- del país para poder comparar países con distinta cantidad de usuarios
    SELECT p.country, p.artist_id, a.artist_name, p.oyentes,
           p.oyentes::DOUBLE / t.n AS pct_pais
    FROM oyentes_pais p

    -- Conservamos los países que cumplen con el mínimo de usuarios
    JOIN usuarios_pais t ON p.country = t.country

    -- Conservamos los artistas con más oyentes
    JOIN mas_oyentes m ON p.artist_id = m.artist_id

    -- Incorporamos el nombre de cada artista
    JOIN artists a ON p.artist_id = a.artist_id
""", "pais_artista")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pais_artista: 458,744 filas guardadas en C:/Users/mgrau/musica-por-pais/data/tablas/pais_artista.parquet


In [12]:
# Cargamos la tabla pais_artista
pais_artista = q(f"SELECT * FROM '{ruta_tablas}pais_artista.parquet'")

# Verificamos la cantidad de filas, países y artistas de la tabla
print("Filas:", f"{len(pais_artista):,}",
      "| Países:", pais_artista["country"].nunique(),
      "| Artistas:", pais_artista["artist_id"].nunique())

# Sumamos los oyentes de cada artista entre los países seleccionados
# y mostramos los 10 artistas con más oyentes
(pais_artista.groupby("artist_name")["oyentes"].sum()
 .sort_values(ascending=False).head(10))

Filas: 458,744 | Países: 56 | Artistas: 10000


artist_name
Daft Punk                19411
Radiohead                18631
Coldplay                 17471
Foo Fighters             16755
Arctic Monkeys           16596
The Cure                 16315
Red Hot Chili Peppers    15930
Kings of Leon            15842
The Killers              15637
System of a Down         15209
Name: oyentes, dtype: int64

### 4.3 Artistas por año

Derivación: obtenemos el año de cada escucha a partir del timestamp, que está expresado en segundos desde 1970, en UTC.

Agregación: calculamos la cantidad de oyentes distintos y de escuchas por artista y año. Para contar una sola vez a cada oyente, primero agrupamos por usuario, artista y año, y luego por año y artista.

Filtrado: seleccionamos los 30 artistas con más oyentes de cada año. Esto nos permite armar un ranking de los 10 primeros y analizar cómo cambia su posición a lo largo del tiempo.

Como 2005 y 2014 no son años completos y la cantidad de usuarios activos varía entre años, conviene comparar la posición de cada artista en el ranking en lugar de la cantidad absoluta de oyentes.

In [13]:
# Armamos un ranking de los artistas con más oyentes de cada año
# para analizar cómo cambia su popularidad a lo largo del tiempo.

guardar("""
    -- Obtenemos el año a partir del timestamp y contamos las escuchas
    -- de cada usuario por artista y año
    WITH por_usuario AS (
        SELECT user_id, artist_id,
               EXTRACT(year FROM to_timestamp("timestamp")) AS anio,
               COUNT(*) AS escuchas
        FROM events_clean
        GROUP BY user_id, artist_id, anio
    ),

    -- Calculamos la cantidad de oyentes distintos y las escuchas
    -- totales de cada artista por año
    por_artista AS (
        SELECT anio, artist_id, COUNT(*) AS oyentes, SUM(escuchas) AS escuchas
        FROM por_usuario
        GROUP BY anio, artist_id
    ),

    -- Ordenamos los artistas de cada año por cantidad de oyentes.
    -- En caso de empate, usamos la cantidad de escuchas
    ranking AS (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY anio ORDER BY oyentes DESC, escuchas DESC
        ) AS puesto
        FROM por_artista
    )

    -- Seleccionamos los datos que queremos guardar
    SELECT r.anio, r.puesto, r.artist_id, a.artist_name, r.oyentes, r.escuchas
    FROM ranking r

    -- Agregamos el nombre del artista
    JOIN artists a ON r.artist_id = a.artist_id

    -- Nos quedamos con los 30 primeros artistas de cada año
    WHERE r.puesto <= 30
""", "artista_anio")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

artista_anio: 300 filas guardadas en C:/Users/mgrau/musica-por-pais/data/tablas/artista_anio.parquet


In [14]:
# Cargamos la tabla artista_anio
artista_anio = q(f"SELECT * FROM '{ruta_tablas}artista_anio.parquet'")

# Verificamos que haya 30 artistas por cada año
print(artista_anio.groupby("anio").size())

# Seleccionamos los 5 artistas con más oyentes de cada año
# y reorganizamos la tabla para comparar los rankings entre años
artista_anio[artista_anio["puesto"] <= 5].pivot(
    index="puesto",
    columns="anio",
    values="artist_name"
)

anio
2005    30
2006    30
2007    30
2008    30
2009    30
2010    30
2011    30
2012    30
2013    30
2014    30
dtype: int64


anio,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014
puesto,,,,,,,,,,
1,The White Stripes,The White Stripes,Radiohead,Radiohead,Radiohead,Radiohead,Radiohead,Radiohead,Daft Punk,Daft Punk
2,Radiohead,Radiohead,The White Stripes,Coldplay,Kings of Leon,Kings of Leon,Foo Fighters,Adele,Radiohead,Arctic Monkeys
3,System of a Down,System of a Down,Foo Fighters,The Killers,Coldplay,Coldplay,Daft Punk,Coldplay,Arctic Monkeys,Coldplay
4,Foo Fighters,Coldplay,The Killers,The Cure,The Killers,Daft Punk,Coldplay,Foo Fighters,The xx,Radiohead
5,Coldplay,Death Cab for Cutie,Arctic Monkeys,Foo Fighters,The Cure,The Cure,Adele,The xx,The Cure,The xx
